# L09 · Assignment — Recipe semantic search

> *Sarah's product search demo went well. Marcus's friend who runs a recipe website asks her for the same thing — but for recipes. The corpus is small (30 recipes) but the queries are even more synonym-heavy: "creamy Italian dessert", "spicy something with rice", "summery starter".*

In this assignment you will:

**Part A.** Build a semantic search engine on a recipe corpus. Apply it to a benchmark of 6 natural-language queries.

**Part B.** Build a TF-IDF baseline and compare. **Target: semantic ≥ TF-IDF on top-1.**

**Part C.** Reflect on **two** specific failure modes and how to fix them.

Total runtime: ~5 minutes.

---

## Setup

In [1]:
# --- Colab setup: install sentence-transformers if missing (no-op in the local dsai-m3 env) ---
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)

import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

torch.set_num_threads(1)
np.random.seed(0)

# Load data — local file if present (VS Code), else fetch from GitHub (works in Colab).
import os
_LOCAL = 'data/recipes.csv'
_URL = 'https://raw.githubusercontent.com/flexfengfeng/6m-data-3.9-Natural-Language-Processing/main/notebooks/data/recipes.csv'
df = pd.read_csv(_LOCAL if os.path.exists(_LOCAL) else _URL)
print(f"Recipes: {len(df)}")
print(f"Categories: {df['category'].value_counts().to_dict()}")
df.head(3)

/opt/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Recipes: 30
Categories: {'pizza': 5, 'pasta': 5, 'salad': 5, 'dessert': 5, 'drink': 5, 'curry': 5}


,recipe_id,name,category,description
0,R001,Classic Margherita,pizza,"Wood-fired pizza with fresh mozzarella, basil ..."
1,R002,Pepperoni Supreme,pizza,Loaded pepperoni pizza with extra cheese and a...
2,R003,Garden Vegetable Pizza,pizza,"Bell peppers, mushrooms, olives, and red onion..."


---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Part A — build the semantic search engine + benchmark |
| **🔵 Advanced Track** | Learners with prior ML background | Part B — TF-IDF baseline comparison + Part C failure-mode reflection |

If you're unsure, start with the **Foundational Track**. If it feels easy, skip ahead to the **Advanced Track** — both tracks cover the same lesson outcomes; only the scaffolding differs.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


---

## Part A — Semantic search

Load the pretrained model, embed every recipe, build a `search()` function.

### A.1 · Embed the corpus

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Build the document text — name + description
documents = (df['name'] + ' — ' + df['description']).tolist()

print(f"Embedding {len(documents)} recipes...")
embeddings = model.encode(documents, show_progress_bar=False)
print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8695.05it/s]

Embedding 30 recipes...


Embeddings shape: (30, 384)


### A.2 · Define the search function

In [3]:
def semantic_search(query, top=5):
    q = model.encode([query])
    sims = cosine_similarity(q, embeddings)[0]
    order = np.argsort(-sims)[:top]
    return df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

# Sanity check
print("Query: 'creamy Italian dessert'")
print(semantic_search('creamy Italian dessert', top=3)[['category','name','score']].to_string(index=False))

Query: 'creamy Italian dessert'
category                name    score
 dessert            Tiramisu 0.589747
   pasta Spaghetti Carbonara 0.585897
   drink          Cappuccino 0.529143


### A.3 · The benchmark queries

Your engine will be evaluated against these 6 queries:

In [4]:
BENCHMARK = {
    "creamy Italian dessert":          "Tiramisu",
    "spicy curry with coconut milk":   "Thai Green Curry",
    "refreshing fruit smoothie":       "Mango Smoothie",
    "Mediterranean salad with feta":   "Greek Village Salad",
    "rich chocolate pudding":          "Chocolate Lava Cake",
    "vegan main with beans":           "Vegan Chickpea Curry",
}

correct = 0
print(f"{'Query':40s} {'Top-1 prediction':30s} {'Expected':25s}")
print('-' * 100)
for q, expected in BENCHMARK.items():
    top1 = semantic_search(q, top=1).iloc[0]['name']
    ok = top1 == expected
    correct += int(ok)
    print(f"  {'✅' if ok else '❌'}  {q:36s} {top1:30s} {expected:25s}")
sem_acc = correct / len(BENCHMARK)
print(f"\n[Part A] Semantic top-1 accuracy: {correct}/{len(BENCHMARK)} = {sem_acc:.0%}")

Query                                    Top-1 prediction               Expected                 
----------------------------------------------------------------------------------------------------
  ✅  creamy Italian dessert               Tiramisu                       Tiramisu                 
  ✅  spicy curry with coconut milk        Thai Green Curry               Thai Green Curry         
  ✅  refreshing fruit smoothie            Mango Smoothie                 Mango Smoothie           
  ✅  Mediterranean salad with feta        Greek Village Salad            Greek Village Salad      
  ✅  rich chocolate pudding               Chocolate Lava Cake            Chocolate Lava Cake      
  ✅  vegan main with beans                Vegan Chickpea Curry           Vegan Chickpea Curry     

[Part A] Semantic top-1 accuracy: 6/6 = 100%


---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. You're welcome to peek at the Foundational Track above for reference.*

---


---

## Part B — TF-IDF baseline

Build a TF-IDF retriever on the same corpus.

In [5]:
vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vec.fit_transform(documents)
print(f"TF-IDF vocab: {len(vec.vocabulary_)}, matrix shape: {tfidf_matrix.shape}")

def tfidf_search(query, top=5):
    q = vec.transform([query])
    sims = cosine_similarity(q, tfidf_matrix)[0]
    order = np.argsort(-sims)[:top]
    return df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

# Run benchmark
correct = 0
print(f"{'Query':40s} {'TF-IDF top-1':30s} {'Expected':25s}")
print('-' * 100)
for q, expected in BENCHMARK.items():
    top1 = tfidf_search(q, top=1).iloc[0]['name']
    ok = top1 == expected
    correct += int(ok)
    print(f"  {'✅' if ok else '❌'}  {q:36s} {top1:30s} {expected:25s}")
tf_acc = correct / len(BENCHMARK)
print(f"\n[Part B] TF-IDF top-1 accuracy: {correct}/{len(BENCHMARK)} = {tf_acc:.0%}")

TF-IDF vocab: 578, matrix shape: (30, 578)
Query                                    TF-IDF top-1                   Expected                 
----------------------------------------------------------------------------------------------------
  ❌  creamy Italian dessert               Chicken Tikka Masala           Tiramisu                 
  ✅  spicy curry with coconut milk        Thai Green Curry               Thai Green Curry         
  ✅  refreshing fruit smoothie            Mango Smoothie                 Mango Smoothie           
  ✅  Mediterranean salad with feta        Greek Village Salad            Greek Village Salad      
  ✅  rich chocolate pudding               Chocolate Lava Cake            Chocolate Lava Cake      
  ❌  vegan main with beans                Niçoise Salad                  Vegan Chickpea Curry     

[Part B] TF-IDF top-1 accuracy: 4/6 = 67%


### B.1 · Head-to-head

In [6]:
print(f"{'Approach':12s} {'Top-1 acc':>10s}")
print('-' * 24)
print(f"{'Semantic':12s} {sem_acc:>10.0%}")
print(f"{'TF-IDF':12s} {tf_acc:>10.0%}")
print()
print(f"Target: Semantic ≥ TF-IDF: {'✅ PASS' if sem_acc >= tf_acc else '❌ FAIL — investigate'}")

Approach      Top-1 acc
------------------------
Semantic           100%
TF-IDF              67%

Target: Semantic ≥ TF-IDF: ✅ PASS


---

## Part C — Reflect on failure modes

Identify **two** queries from the benchmark where semantic search failed (or where its top-1 wasn't ideal), and explain:

1. **Why** did the model rank the wrong recipe first?
2. **What** could you do to fix it in a production system?

(If semantic got all 6 right, pick the **two least confident** queries — the ones with the lowest top-1 score — and reflect on what would happen if the corpus was 10× bigger and more diverse.)

### C.1 · Identify candidates for failure

In [7]:
# For each query, show top-3 results and their scores
for q, expected in BENCHMARK.items():
    print(f"\nQuery: {q!r}  (expected: {expected})")
    top3 = semantic_search(q, top=3)
    print(top3[['name','category','score']].to_string(index=False))


Query: 'creamy Italian dessert'  (expected: Tiramisu)


               name category    score
           Tiramisu  dessert 0.589747
Spaghetti Carbonara    pasta 0.585897
         Cappuccino    drink 0.529143

Query: 'spicy curry with coconut milk'  (expected: Thai Green Curry)


               name category    score
   Thai Green Curry    curry 0.624226
     Massaman Curry    curry 0.568293
Japanese Beef Curry    curry 0.519857

Query: 'refreshing fruit smoothie'  (expected: Mango Smoothie)
             name category    score
   Mango Smoothie    drink 0.655978
    Apple Crumble  dessert 0.414793
Iced Matcha Latte    drink 0.379388

Query: 'Mediterranean salad with feta'  (expected: Greek Village Salad)
               name category    score
Greek Village Salad    salad 0.661248
       Caesar Salad    salad 0.585490
 Classic Margherita    pizza 0.495310

Query: 'rich chocolate pudding'  (expected: Chocolate Lava Cake)
               name category    score
Chocolate Lava Cake  dessert 0.466780
       Crème Brûlée  dessert 0.439637
      Apple Crumble  dessert 0.426559

Query: 'vegan main with beans'  (expected: Vegan Chickpea Curry)
                  name category    score
  Vegan Chickpea Curry    curry 0.468695
         Niçoise Salad    salad 0.335845
Garden V

### C.2 · Your reflection

Pick two queries. For each, write 2-3 sentences explaining:

- WHY the model behaved as it did
- WHAT signal (filter, re-ranker, additional context) would fix it in production

**Query 1 reflection:** *(your answer)*

**Query 2 reflection:** *(your answer)*

### C.3 · Open-ended improvement

Your semantic engine likely scored **100% top-1** on these six (fairly easy) queries. Now imagine a larger, noisier corpus where it only reached ~80%: if you had to add ONE thing to push it from 80% to 95% top-1, what would it be? (one sentence)

*(your answer)*

---

## Submission checklist

- [ ] Part A — Semantic search built, benchmarked, accuracy printed
- [ ] Part B — TF-IDF baseline built, head-to-head comparison printed
- [ ] Part B target — Semantic ≥ TF-IDF (PASS)
- [ ] Part C — Two failure-mode reflections completed
- [ ] Part C — One open-ended improvement suggestion

Save the notebook with outputs and submit.